In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
.appName("Analisis_Categorias") \
.config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata.datos_libros") \
.config("spark.jars.packages","org.mongodb.spark:mongo-spark-connector_2.12:10.3.0") \
.getOrCreate()

df = spark.read.format("mongodb").load()

df.printSchema()
df.show(5)



root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+--------------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|               grupo|                item|valor|
+--------------------+---------+-------------------+--------------------+--------------------+-----+
|69d65b575eb6b44d5...|   Travel|2026-04-08 13:42:47|        G1_Ecommerce|  It's Only Himalaya|45.17|
|69d65b575eb6b44d5...|   Travel|2026-04-08 13:42:47|        G1_Ecommerce|Full Moon over No...|49.43|
|69defbab676143461...|   Travel|2026-04-15 02:44:59|G5_Turismo_y_Host...|  It's Only Himalaya|45.17|
|69defbab676143461...|   Travel|2026-04-15 02:44:59|G5_Turismo_y_Host...|Full Moon over No...|49.43|
|69df8180676143461...|   Travel|2026-04-15 12:16:00|G5_T

In [3]:
from pyspark.sql.functions import col

df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 3: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 3: Limpieza completada.
Registros originales: 6
Registros validos: 6


In [4]:
df_filtrado.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
|  It's Only Himalaya|45.17|
|Full Moon over No...|49.43|
+--------------------+-----+



In [5]:
df_filtrado.filter(df_filtrado["valor"] > 100).show()

+---+---------+-------------+-----+----+-----+
|_id|categoria|fecha_captura|grupo|item|valor|
+---+---------+-------------+-----+----+-----+
+---+---------+-------------+-----+----+-----+



In [6]:
df_filtrado.groupBy("grupo").count().show()

+--------------------+-----+
|               grupo|count|
+--------------------+-----+
|G5_Turismo_y_Host...|    4|
|        G1_Ecommerce|    2|
+--------------------+-----+



In [7]:
from pyspark.sql import functions as F

reporte_categorias = df_filtrado.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+---------+---------------+------------------+-------------+-------------+
|categoria|cantidad_libros|   precio_promedio|precio_minimo|precio_maximo|
+---------+---------------+------------------+-------------+-------------+
|   Travel|              6|47.300000000000004|        45.17|        49.43|
+---------+---------------+------------------+-------------+-------------+

